# Case Study 2 — EfficientNetB0 Inference & Visualization
**Loads the already-trained model and shows predictions inline — no retraining needed.**

In [1]:
import json, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as PIL_Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

print('✔ Libraries loaded')

C:\Users\prana\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


✔ Libraries loaded


In [2]:
BASE_DIR   = Path(r'c:\Users\prana\OneDrive\Desktop\SUB\SEM 6\NNDL\NNDK')
DATA_DIR   = BASE_DIR / 'archive_1' / 'PlantVillage'
CS2_DIR    = BASE_DIR / 'Case study 2'
OUTPUT_DIR = CS2_DIR / 'outputs'
MODEL_PATH = OUTPUT_DIR / 'models' / 'tomato3class_efficientnet_final.keras'
VIZ_DIR    = OUTPUT_DIR / 'predictions'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

IMG_H = 224; IMG_W = 224; SEED = 42; BATCH = 16
SAMPLES_PER_CLASS = 1000
VIZ_PER_CLS = 3   # show 3 samples per class inline
TARGET_CLASSES = ['Tomato_Bacterial_spot','Tomato_Early_blight','Tomato_Late_blight']

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Load saved model
model = tf.keras.models.load_model(str(MODEL_PATH))
print(f'✔ Model loaded from: {MODEL_PATH}')

# Class maps
classes     = sorted(TARGET_CLASSES)
num_classes = len(classes)
name_to_idx = {n: i for i, n in enumerate(classes)}
idx_to_name = {i: n for n, i in name_to_idx.items()}
print(f'Classes: {classes}')

TypeError: Could not locate class 'WeightedFocalLoss'. Make sure custom classes and functions are decorated with `@keras.saving.register_keras_serializable()`. If they are already decorated, make sure they are all imported so that the decorator is run before trying to load them. Full object config: {'module': None, 'class_name': 'WeightedFocalLoss', 'config': {'name': 'weighted_focal_loss', 'reduction': 'sum_over_batch_size', 'class_weights': [1.0, 1.0, 1.0], 'gamma': 2.0}, 'registered_name': 'WeightedFocalLoss'}

In [ ]:
# Build same balanced test split (same seed → same 450 test images)
valid_exts = {'.jpg','.jpeg','.png','.bmp'}
image_paths, labels = [], []
for cls_name in classes:
    cls_dir   = DATA_DIR / cls_name
    cls_files = [str(f) for f in cls_dir.iterdir() if f.suffix.lower() in valid_exts]
    if len(cls_files) > SAMPLES_PER_CLASS:
        cls_files = random.sample(cls_files, SAMPLES_PER_CLASS)
    image_paths.extend(cls_files)
    labels.extend([name_to_idx[cls_name]] * len(cls_files))

image_paths = np.array(image_paths)
labels      = np.array(labels, dtype=np.int32)

_, X_temp, _, y_temp = train_test_split(image_paths, labels, test_size=0.30, random_state=SEED, stratify=labels)
X_test, y_test, *_  = [*train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)]
print(f'Test set: {len(y_test)} images ({len(y_test)//num_classes} per class)')

In [ ]:
# Run inference on test set
def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_H, IMG_W))
    return tf.cast(img, tf.float32)

AUTOTUNE = tf.data.AUTOTUNE
test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .map(lambda p, l: (load_img(p), l), num_parallel_calls=AUTOTUNE)
           .batch(BATCH).prefetch(AUTOTUNE))

test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f'\n✔ Test Accuracy : {test_acc*100:.2f}%')
print(f'✔ Test Loss     : {test_loss:.4f}')

y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
print('\n📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=[idx_to_name[i] for i in range(num_classes)]))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
short = [idx_to_name[i].replace('Tomato_','') for i in range(num_classes)]

fig, ax = plt.subplots(figsize=(8,6), facecolor='#0D1117')
ax.set_facecolor('#0D1117')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short, yticklabels=short, ax=ax,
            linewidths=0.5, linecolor='#333', annot_kws={'size':14})
ax.set_title('Confusion Matrix — Test Set', color='white', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Predicted', color='white', fontsize=11)
ax.set_ylabel('Actual',    color='white', fontsize=11)
ax.tick_params(colors='white', labelsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 🔬 Per-Sample Prediction Cards (3 per class)

In [ ]:
BAR_COLORS = ['#2196F3','#FF9800','#4CAF50']
class_to_idx = defaultdict(list)
for i, lbl in enumerate(y_test):
    class_to_idx[lbl].append(i)

for cls_idx in range(num_classes):
    cls_name  = idx_to_name[cls_idx]
    cls_short = cls_name.replace('Tomato_','')
    samples   = random.sample(class_to_idx[cls_idx], min(VIZ_PER_CLS, len(class_to_idx[cls_idx])))

    print(f'\n--- {cls_name} ---')
    for rank, idx in enumerate(samples):
        img_path   = X_test[idx]
        true_lbl   = int(y_test[idx])
        pred_lbl   = int(y_pred[idx])
        probs      = y_pred_probs[idx]
        confidence = float(probs[pred_lbl])
        correct    = (true_lbl == pred_lbl)

        img_arr = np.array(PIL_Image.open(img_path).convert('RGB').resize((IMG_H, IMG_W)))
        status  = '✅ CORRECT' if correct else '❌ WRONG'
        s_color = '#4CAF50' if correct else '#F44336'

        fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11, 4),
                                              gridspec_kw={'width_ratios':[1,1.4]},
                                              facecolor='#0D1117')

        ax_img.imshow(img_arr); ax_img.axis('off')
        ax_img.set_title(
            f'{status}\nTrue : {cls_name.replace("Tomato_","")}\n'
            f'Pred : {idx_to_name[pred_lbl].replace("Tomato_","")}',
            color=s_color, fontsize=9, fontweight='bold', pad=6)

        ax_bar.set_facecolor('#161B22')
        snames = [idx_to_name[i].replace('Tomato_','') for i in range(num_classes)]
        bars = ax_bar.barh(snames, probs*100, color=BAR_COLORS, edgecolor='#333', height=0.5)
        for bar, p in zip(bars, probs):
            ax_bar.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                        f'{p*100:.1f}%', va='center', color='white', fontsize=10, fontweight='bold')
        ax_bar.set_xlim(0, 118)
        ax_bar.axvline(x=50, color='#555', linewidth=0.8, linestyle='--')
        ax_bar.set_xlabel('Confidence (%)', color='white', fontsize=9)
        ax_bar.tick_params(colors='white', labelsize=9)
        ax_bar.set_title('EfficientNetB0 — Softmax Output', color='white', fontsize=9, pad=6)
        for spine in ax_bar.spines.values(): spine.set_edgecolor('#444')

        fig.suptitle(
            f'EfficientNet Prediction | {cls_name} | Conf: {confidence*100:.1f}%',
            fontsize=11, color=s_color, fontweight='bold', y=1.02)
        plt.tight_layout()

        out = f'{cls_short}_sample{rank+1}.png'
        plt.savefig(str(VIZ_DIR/out), dpi=130, bbox_inches='tight', facecolor='#0D1117')
        plt.show()
        plt.close()
        print(f"  {'✅' if correct else '❌'} {cls_short} — conf={confidence*100:.1f}%")

## 📊 Training Curves (saved from training)

In [ ]:
import pandas as pd
p1 = pd.read_csv(str(OUTPUT_DIR / 'logs' / 'phase1_log.csv'))
p2 = pd.read_csv(str(OUTPUT_DIR / 'logs' / 'phase2_log.csv'))
hist_acc  = list(p1['accuracy'])  + list(p2['accuracy'])
hist_vacc = list(p1['val_accuracy']) + list(p2['val_accuracy'])
hist_loss = list(p1['loss']) + list(p2['loss'])
hist_vloss= list(p1['val_loss']) + list(p2['val_loss'])
ep        = range(1, len(hist_acc)+1)
p1_end    = len(p1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0D1117')
fig.suptitle('EfficientNetB0 + Weighted Focal Loss — Training History (Balanced 1000/class)',
             fontsize=12, color='white', fontweight='bold')
for ax in axes:
    ax.set_facecolor('#161B22'); ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for s in ax.spines.values(): s.set_edgecolor('#444')
    ax.axvline(x=p1_end+0.5, color='#FFD700', lw=1.5, ls='--', label='Phase 1→2')

axes[0].plot(ep, hist_acc,  label='Train', color='#2196F3', lw=2)
axes[0].plot(ep, hist_vacc, label='Val',   color='#FF9800', lw=2)
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(facecolor='#0D1117', labelcolor='white'); axes[0].grid(True, alpha=0.2)

axes[1].plot(ep, hist_loss,  label='Train', color='#4CAF50', lw=2)
axes[1].plot(ep, hist_vloss, label='Val',   color='#F44336', lw=2)
axes[1].set_title('Focal Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(facecolor='#0D1117', labelcolor='white'); axes[1].grid(True, alpha=0.2)

plt.tight_layout(); plt.show()
print(f'Phase 1: {len(p1)} epochs | Phase 2: {len(p2)} epochs | Total: {len(hist_acc)} epochs')

In [ ]:
# Final summary
print('='*55)
print('  🏆  FINAL RESULTS — EfficientNetB0')
print('='*55)
print(f'  Dataset    : 1000 images/class (balanced)')
print(f'  Loss       : Weighted Focal Loss (gamma=2)')
print(f'  Test Acc   : {test_acc*100:.2f}%')
print(f'  Test Loss  : {test_loss:.4f}')
print('='*55)